# 03. Rede Neural (MLP) — Classificação Binária

Este notebook implementa um modelo de Rede Neural do tipo Perceptron Multicamadas (MLP) para prever se uma escola da Paraíba atinge ou não a meta do IDEB, utilizando as 8 features selecionadas no notebook anterior.

A construção do modelo segue rigorosamente as técnicas ensinadas na disciplina de Aprendizagem de Máquina:

1. **Dimensionamento da arquitetura** pela Regra de Ouro da generalização (dimensão VC).
2. **Seleção de hiperparâmetros** pelo algoritmo de seleção via conjunto de validação.
3. **Regularização L2** para controle de overfitting.
4. **Análise de overfitting** pelo gráfico de E_in vs E_out ao longo das épocas.
5. **Avaliação final** no conjunto de teste com métricas de classificação.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Importar funções do pacote src
sys.path.insert(0, os.path.abspath('..'))
from src.rede_neural import calcular_arquitetura_mlp, criar_modelo_mlp, grid_search_mlp

# Configurações
sns.set_theme(style='whitegrid')
DADOS_TRATADOS = os.path.abspath('../dados_tratados')
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f'TensorFlow: {tf.__version__}')
print('Imports carregados com sucesso.')

---
## 1. Carregamento dos Dados

Os conjuntos de treino e teste foram gerados no notebook 02 (limpeza e EDA), já normalizados com StandardScaler e contendo apenas as 8 features selecionadas por correlação.

In [ ]:
X_train_full = pd.read_csv(os.path.join(DADOS_TRATADOS, 'X_train.csv')).values
X_test = pd.read_csv(os.path.join(DADOS_TRATADOS, 'X_test.csv')).values
y_train_full = pd.read_csv(os.path.join(DADOS_TRATADOS, 'y_train.csv')).values.ravel()
y_test = pd.read_csv(os.path.join(DADOS_TRATADOS, 'y_test.csv')).values.ravel()

N = X_train_full.shape[0]
p = X_train_full.shape[1]

print(f'N (amostras de treino): {N}')
print(f'p (features): {p}')
print(f'Amostras de teste: {X_test.shape[0]}')
print(f'Distribuição do alvo (treino): 1={int(y_train_full.sum())}, 0={int(N - y_train_full.sum())}')

---
## 2. Definição da Arquitetura pela Teoria da Generalização

### 2.1 Dimensão VC e Número de Pesos

A **dimensão VC** (Vapnik-Chervonenkis) de uma rede neural é aproximada pelo seu número total de parâmetros ajustáveis (pesos e biases): $d_{VC} \approx |W|$. Essa é uma aproximação prática amplamente utilizada, pois o cálculo exato da dimensão VC para redes neurais é intratável.

### 2.2 Regra de Ouro da Generalização

Conforme ensinado na disciplina, a **Regra de Ouro** estabelece que o número de amostras de treino deve ser pelo menos 10 vezes a dimensão VC do modelo para garantir boa capacidade de generalização:

$$N \geq 10 \cdot d_{VC} \quad \Longrightarrow \quad N \geq 10 \cdot |W|$$

### 2.3 Escolha de Uma Camada Escondida (Teorema da Aproximação Universal)

Optamos por uma rede com **uma única camada escondida**, fundamentados no **Teorema da Aproximação Universal** (Cybenko, 1989; Hornik, 1991): uma rede feedforward com uma única camada escondida contendo um número finito de neurônios já é suficiente para aproximar qualquer função contínua em um subconjunto compacto de $\mathbb{R}^n$. Adicionar mais camadas aumentaria a dimensão VC sem necessidade teórica, dificultando a generalização.

### 2.4 Cálculo do Número Máximo de Neurônios

Para uma rede com **uma camada escondida de $n$ neurônios**, $d$ entradas e 1 saída, o total de pesos é:

$$|W| = (d + 1) \cdot n + (n + 1)$$

onde o $+1$ em cada termo corresponde ao **bias** de cada neurônio. Substituindo na Regra de Ouro $N \geq 10 \cdot |W|$ e isolando $n$:

$$N \geq 10 \cdot [(d+1)n + (n+1)]$$
$$N \geq 10(d+1)n + 10n + 10$$
$$N - 10 \geq 10n(d + 2)$$

$$\boxed{n \leq \frac{N - 10}{10(d + 2)}}$$

### 2.5 Substituição com os Valores Reais

Com $N = 2674$ amostras de treino e $d = p = 8$ features:

$$n \leq \frac{2674 - 10}{10 \cdot (8 + 2)} = \frac{2664}{100} = 26{,}64$$

O máximo permitido pela Regra de Ouro é $n = 26$ neurônios. Escolhemos **$n = 26$** (o valor inteiro máximo), pois nosso dataset, embora moderado, oferece margem suficiente.

### 2.6 Verificação de |W|

$$|W| = (8 + 1) \times 26 + (26 + 1) = 234 + 27 = 261$$

Conferindo a Regra de Ouro: $N \geq 10 \cdot |W| \Rightarrow 2674 \geq 2610$ ✅

### 2.7 Arquitetura Final

| Camada | Neurônios | Ativação | Parâmetros |
|---|---|---|---|
| Entrada | 8 | — | — |
| Escondida | 26 | ReLU | $(8+1) \times 26 = 234$ |
| Saída | 1 | Sigmoid | $(26+1) \times 1 = 27$ |
| **Total** | | | **261** |

A ativação **sigmoid** na saída é a escolha padrão para classificação binária, conforme ensinado: "classificação binária usa função logística/sigmoid e entropia cruzada binária como métrica de erro".

In [ ]:
# Cálculo da arquitetura pela Regra de Ouro
arq = calcular_arquitetura_mlp(N=N, d=p)

print(f'N (amostras de treino):          {arq["N"]}')
print(f'd (features de entrada):         {arq["d"]}')
print(f'n_max (teórico):                 {arq["n_max_float"]:.2f}')
print(f'n (escolhido):                   {arq["n_escolhido"]}')
print(f'|W| (total de pesos):            {arq["total_pesos"]}')
print(f'10 * |W|:                        {10 * arq["total_pesos"]}')
print(f'Regra de Ouro satisfeita (N>=10|W|): {arq["regra_satisfeita"]}')

n_neuronios = arq['n_escolhido']

---
## 3. Separação do Conjunto de Validação

Conforme a Regra de Ouro para o tamanho do conjunto de validação ensinada no capítulo de Validação da disciplina, o tamanho ideal de $K$ (número de amostras de validação) é dado por:

$$K \approx \frac{N}{5}$$

Ou seja, aproximadamente **20% do treino** é reservado para validação. Esse valor balanceia o dilema entre ter um conjunto de validação grande o suficiente para estimar o erro de forma confiável (estimativa $E_{val}$ estável) e não reduzir excessivamente o conjunto de treino disponível para o aprendizado do modelo.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=SEED, stratify=y_train_full
)

print(f'D_train (treino):     {X_train.shape[0]} amostras')
print(f'D_val (validação):    {X_val.shape[0]} amostras (K ≈ N/5 = {N}//5 = {N//5})')
print(f'D_test (teste final): {X_test.shape[0]} amostras')

---
## 4. Seleção de Hiperparâmetros (Grid Search com Validação)

Seguimos o **algoritmo de seleção de modelos** ensinado no capítulo de Validação: define-se $M$ modelos candidatos com hiperparâmetros diferentes, treina-se cada modelo em $D_{train}$, avalia-se o erro de validação $E_{val}$ em $D_{val}$, e seleciona-se o modelo com **menor $E_{val}$**.

### Hiperparâmetros avaliados

- **Taxa de aprendizado ($\eta$):** Controla o tamanho do passo na descida do gradiente. Valores testados: 0.001, 0.01, 0.1.
- **Tamanho do batch:** Conforme ensinado na disciplina, existem três abordagens para a descida do gradiente:
  - **Estocástica** ($batch\_size = 1$): atualiza os pesos a cada amostra individual — ruidosa mas com atualizações frequentes.
  - **Batch completo** ($batch\_size = N$): calcula o gradiente sobre todo o conjunto de treino — mais estável mas lenta.
  - **Mini-batch** ($1 < batch\_size < N$): compromisso entre as duas, é a mais usada na prática.
  Testamos valores de mini-batch: 16, 32, 64.

### Regularização L2

A regularização L2 adiciona uma penalidade proporcional ao quadrado dos pesos à função de custo:

$$E_{reg} = E_{in} + \lambda \sum_{i} w_i^2$$

onde $\lambda$ é o coeficiente de regularização. Essa penalidade controla a magnitude dos pesos, e embora não altere a dimensão VC teórica do modelo, **reduz a dimensão VC efetiva** ao restringir o espaço de hipóteses efetivamente alcançável durante o treino. Isso ajuda a prevenir overfitting, especialmente quando o dataset é moderado.

Incluímos $\lambda$ como dimensão adicional no grid search, testando: 0.0 (sem regularização), 0.001, 0.01.

In [ ]:
# Definição do grid de hiperparâmetros
learning_rates = [0.001, 0.01, 0.1]
batch_sizes = [16, 32, 64]
l2_lambdas = [0.0, 0.001, 0.01]

total_combinacoes = len(learning_rates) * len(batch_sizes) * len(l2_lambdas)
print(f'Total de combinações a testar: {total_combinacoes}')
print(f'Épocas por modelo: 50')
print(f'Arquitetura fixa: {p} → {n_neuronios} → 1')
print()

df_grid = grid_search_mlp(
    X_train, y_train, X_val, y_val,
    d=p, n=n_neuronios,
    learning_rates=learning_rates,
    batch_sizes=batch_sizes,
    l2_lambdas=l2_lambdas,
    epochs=50,
    verbose=0
)

print(f'\n=== RESULTADO DO GRID SEARCH (Top 10) ===')
print(df_grid.head(10).to_string(index=False))

# Melhor combinação
melhor = df_grid.iloc[0]
print(f'\n🏆 MELHOR COMBINAÇÃO:')
print(f'  Taxa de aprendizado: {melhor["learning_rate"]}')
print(f'  Batch size: {int(melhor["batch_size"])}')
print(f'  L2 lambda: {melhor["l2_lambda"]}')
print(f'  E_val (loss): {melhor["val_loss"]:.4f}')
print(f'  Acurácia validação: {melhor["val_accuracy"]:.4f}')

---
## 5. Treinamento com Melhores Hiperparâmetros (Análise de Overfitting)

Com os hiperparâmetros vencedores selecionados pelo grid search, treinamos o modelo por um número alto de épocas (150) para observar o comportamento completo das curvas de aprendizado ($E_{in}$ vs $E_{out}$) e identificar o ponto a partir do qual ocorre overfitting.

In [ ]:
# Retreinar com os melhores hiperparâmetros por 150 épocas
best_lr = melhor['learning_rate']
best_bs = int(melhor['batch_size'])
best_l2 = melhor['l2_lambda']

tf.random.set_seed(SEED)
np.random.seed(SEED)

modelo_analise = criar_modelo_mlp(d=p, n=n_neuronios, learning_rate=best_lr, l2_lambda=best_l2)

history = modelo_analise.fit(
    X_train, y_train,
    epochs=150,
    batch_size=best_bs,
    validation_data=(X_val, y_val),
    verbose=0
)

print('Treinamento concluído (150 épocas).')

### 5.1 Gráfico de Overfitting: E_in vs E_out por Época

O gráfico abaixo mostra a evolução da loss de treino ($E_{in}$) e da loss de validação ($E_{out}$) ao longo das épocas. O ponto de overfitting é identificado como o momento em que a $E_{out}$ (val_loss) começa a subir enquanto a $E_{in}$ (loss de treino) continua caindo — indicando que o modelo começa a memorizar os dados de treino em vez de generalizar.

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(train_loss) + 1)

# Encontrar a época com menor val_loss
best_epoch = np.argmin(val_loss) + 1
best_val_loss = min(val_loss)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, train_loss, label='$E_{in}$ (Loss Treino)', color='#3498db')
ax.plot(epochs_range, val_loss, label='$E_{out}$ (Loss Validação)', color='#e74c3c')
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.7,
           label=f'Melhor época: {best_epoch} (val_loss={best_val_loss:.4f})')
ax.set_xlabel('Época')
ax.set_ylabel('Loss (Binary Crossentropy)')
ax.set_title('Análise de Overfitting: $E_{in}$ vs $E_{out}$')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(DADOS_TRATADOS, 'nn_overfitting.png'), bbox_inches='tight')
plt.show()

print(f'Época com menor E_val: {best_epoch}')
print(f'E_val mínimo: {best_val_loss:.4f}')

---
## 6. Treinamento do Modelo Final

Com base na análise do gráfico de overfitting, utilizamos o callback **EarlyStopping** do Keras para interromper o treinamento automaticamente quando a loss de validação parar de melhorar. O parâmetro `patience` define quantas épocas de "tolerância" o algoritmo aguarda antes de parar, e `restore_best_weights=True` garante que os pesos finais são os da época com menor $E_{val}$ (e não os da última época executada).

Optamos por treinar novamente usando `X_train` e `y_train` (sem reincorporar o conjunto de validação), pois o EarlyStopping requer dados de validação para funcionar como critério de parada. Reincorporar a validação ao treino exigiria fixar o número de épocas manualmente, o que é menos robusto.

In [ ]:
# Treinar modelo final com EarlyStopping
tf.random.set_seed(SEED)
np.random.seed(SEED)

modelo_final = criar_modelo_mlp(d=p, n=n_neuronios, learning_rate=best_lr, l2_lambda=best_l2)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

history_final = modelo_final.fit(
    X_train, y_train,
    epochs=300,
    batch_size=best_bs,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=0
)

epoca_final = len(history_final.history['loss'])
epoca_restaurada = epoca_final - early_stop.patience if early_stop.stopped_epoch > 0 else epoca_final
print(f'Épocas executadas: {epoca_final}')
print(f'Pesos restaurados da melhor época')

modelo_final.summary()

---
## 7. Métricas Finais no Conjunto de Teste

Somente nesta etapa utilizamos o conjunto de teste (`X_test`, `y_test`), que esteve completamente isolado durante todo o processo de treinamento e seleção de hiperparâmetros. Essa separação garante uma avaliação honesta da capacidade de generalização do modelo.

In [ ]:
# Previsões
y_pred_prob = modelo_final.predict(X_test, verbose=0).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

# Métricas
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('=== MÉTRICAS NO CONJUNTO DE TESTE ===')
print(f'Acurácia:  {acc:.4f}')
print(f'Precisão:  {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-Score:  {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Não Atingiu (0)', 'Atingiu (1)']))

In [ ]:
# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Não Atingiu (0)', 'Atingiu (1)'],
            yticklabels=['Não Atingiu (0)', 'Atingiu (1)'])
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão — Rede Neural (MLP)')
plt.tight_layout()
plt.savefig(os.path.join(DADOS_TRATADOS, 'nn_confusion_matrix.png'), bbox_inches='tight')
plt.show()

---
## 8. Salvamento do Modelo

In [ ]:
# Criar diretório e salvar
modelo_dir = os.path.abspath('../modelos')
os.makedirs(modelo_dir, exist_ok=True)
modelo_path = os.path.join(modelo_dir, 'rede_neural.keras')
modelo_final.save(modelo_path)
print(f'Modelo salvo em: {modelo_path}')

---
## 9. Resumo Final (Rede Neural)

### Arquitetura

| Propriedade | Valor |
|---|---|
| Tipo | MLP (Perceptron Multicamadas) |
| Camada de entrada | 8 neurônios (features) |
| Camada escondida | 26 neurônios, ativação ReLU |
| Camada de saída | 1 neurônio, ativação Sigmoid |
| Total de pesos |W| | 261 |
| Regra de Ouro (N ≥ 10|W|) | 2674 ≥ 2610 ✅ |

### Hiperparâmetros

In [ ]:
print('=== RESUMO COMPLETO DA REDE NEURAL ===')
print()
print('--- Arquitetura ---')
print(f'  Camadas: {p} → {n_neuronios} → 1')
print(f'  |W| (total de pesos): {arq["total_pesos"]}')
print(f'  Regra de Ouro: N={N} ≥ 10×|W|={10*arq["total_pesos"]} ✅')
print()
print('--- Hiperparâmetros (vencedores do Grid Search) ---')
print(f'  Taxa de aprendizado (η): {best_lr}')
print(f'  Batch size: {best_bs}')
print(f'  Regularização L2 (λ): {best_l2}')
print(f'  Época final: {epoca_final} (com EarlyStopping, patience=15)')
print()
print('--- Métricas no Conjunto de Teste ---')
print(f'  Acurácia:  {acc:.4f}')
print(f'  Precisão:  {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'  F1-Score:  {f1:.4f}')